# Biblioteka do automatycznego różniczkowania

Kacper Aleksander 325446

In [7]:
using LinearAlgebra

abstract type Operator end # abstrakcyjny typ Operator
const Chain = Vector{Operator} # alias Chain do Vector{Operator}
abstract type Loss end # abstrakcyjny typ Loss

# Tensor
struct Tensor{N}
  # struktura Tensor z parametrem N, który jest przekazywany do NTuple. Czyli Tensor N
  # składa się z atrybutu outsize, który jest typu NTuple o długości N i każdy z jego
  # elementów ma typ Int64 
  outsize :: NTuple{N, Int64}
end
  tensor(sz...) = Tensor(sz)()
# ... to tzw. splat operator, jak *args w Pythonie. Można wtedy pisać tensor(2, 3, 4), co spowoduje spakowanie
# tych wartości do krotki: sz = (2, 3, 4) i uruchomienie konstruktora Tensor(sz). Krotka sz zostanie przekształcona
# na NTuple{N, Int64} automatycznie, aby dopasować się do definicji atrybutu "outsize" Tensora.

function (x::Tensor{N})() where N
  data = zeros(x.outsize...)
  return GraphNode(data)
end

function chain(operators)
# chain to funkcja tworząca wektor operatorów. Wejściem jest wektor składający się z pojedynczych
# operatorów lub krotek operatorów - w tym drugim przypadku są one odpakowywane
  function flatten(x::Tuple)
    y = Vector{Operator}()
    for v in x
      if v isa Tuple
        push!(y, v...)
      else
        push!(y, v)
      end
      end
    return y
  end

  result = Vector{Operator}()
  for operator in flatten(operators)
    push!(result, operator)
  end
  return result
end

function (chain::Chain)(x)
# (chain::Chain) oznacza, że jest to funkcja nieprzypisana do nazwy, a do każdego z obiektów
# typu Chain. x to argument.
  node = x # wartość początkowa
  for op in chain
    node = op(node) # dla każdej operacji w chainie (wektorze operacji) wykonaj tę operację na wartości node.
  end
  return node # zwróć uzyskaną wartość
end
mutable struct GraphNode{OP, N}
  args :: NTuple{N, GraphNode}
  grad
  data
end

const GraphWeight = GraphNode{:weight, 0} # Węzeł bez argumentów gdzie OP = :weight
const GraphTensor = GraphNode{:tensor, 0} # Węzeł bez argumentów gdzie OP = :tensor

# Konstruktor dla danych jak wagi i tensory (obrazy), czyli nieposiadających argumentów (rodziców):
function GraphNode(data::T, trainable=false) where T
  if trainable # wagi są trainable
    return GraphNode{:weight, 0}((), zero(data), data)
  else # tensory nie są trainable
    return GraphNode{:tensor, 0}((), zero(data), data)
  end
end

# Konstruktor dla operatorów-symboli, czyli operatorów, które zaczynają się od :, np. :mul.
# Wymaga podania argumentów! (args)
function GraphNode(op::Symbol, args::Tuple, data::T) where T
  N = length(args)
  grad = similar(data)
  return GraphNode{op, N}(args, grad, data)
end

# Zwracanie wektora GraphNode'ów w kolejności:
function graph(node)
  function visit!(node::GraphNode, visited, ordered)
    if node in visited
    else # jeśli node nie był odwiedzony
      push!(visited, node) # dodaj node do odwiedzonych
      for arg in node.args # dla każdego argumentu w node (jeżeli są)
        visit!(arg, visited, ordered)
        # Odwiedź argumenty ZANIM dodasz obecny node do listy.
        # Gwarantuje to odpowiednią kolejność w grafie. 
      end
      push!(ordered, node)
    end
    return nothing
  end
  ordered = Vector{GraphNode}() # utwórz wektor gdzie każdy element to GraphNode
  visited = Set{GraphNode}() # utwórz wektor z odwiedzonymi GraphNode'ami
  visit!(node, visited, ordered) # odwiedź node i modyfikuj visited oraz ordered
  return ordered # zwróć uporządkowany wektor GraphNode'ów
end

# Funkcja do wyzerowywania gradientu w całym wektorze GraphNode'ów:
function zerograd!(order :: Vector{GraphNode})
  for node in order
    node.grad .= 0
  end
end

# Funkcje, które nic nie robią, ale zapobiegają błędowi kompilatora.
# primal! liczy wartość przejścia w przód - ale nie trzeba jej liczyć dla GraphTensor
# i GraphWeight, ponieważ te wartości są ustalone.
# adjoint! liczy gradient.
function primal!(tensor::GraphTensor) end
function primal!(weight::GraphWeight) end
function adjoint!(::GraphTensor) end
function adjoint!(::GraphWeight) end

# Funkcja forward! przyjmuje posortowany wektor GraphNode'ów oraz pary
# i inicjalizuje data w GraphNode'ach faktycznymi wartościami, a następnie
# przechodzi po Vector{GraphNode} i liczy wartości za pomocą funkcji określonych
# w sekcji Operatory.
function forward!(order::Vector{GraphNode}, pairs...)
  for pair in pairs
    tensor,data = pair
    tensor.data .= data
  end

  for node in order
    primal!(node)
  end
end

# Funkcja backward! liczy gradient.
function backward!(order::Vector{GraphNode})
	seed = last(order)
	seed.grad .= 1

  for node in reverse(order) # w przeciwnym kierunku niż forward!
    adjoint!(node)
  end
end

# Algorytm spadku gradientu
function optimize!(graph, η)
  for node in graph
	if node isa GraphWeight
      node.data .-= η * node.grad
    end
  end
end

# Wyświetlanie
import Base: show
show(io::IO, x::GraphNode{OP, N}) where {OP,N} =
  print(io, "layer ", OP, " with ", N, " arg(s)")
show(io::IO, x::GraphWeight) = print(io, "weight")
show(io::IO, x::GraphTensor) = print(io, "tensor")

## Operatory

In [ ]:
# Add

function primal!(z::GraphNode{:add, 2})
  x, y = z.args
  z.data = x.data .+ y.data
  return nothing
end

function adjoint!(z::GraphNode{:add, 2})
  x, y = z.args
  x.grad += z.grad
  y.grad += z.grad
  return nothing
end

# Sum

function primal!(y::GraphNode{:sum, 1})
  x, = y.args
  y.data = sum(x.data)
  return nothing
end

function adjoint!(y::GraphNode{:sum, 1})
  x, = y.args
  x.grad += y.grad
  return nothing
end

# Mul

function primal!(y::GraphNode{:mul, 2})
  W, x = y.args
  y.data = W.data * x.data
  return nothing
end

function adjoint!(y::GraphNode{:mul, 2})
  W, x = y.args
  W.grad += y.grad * x.data'
  x.grad += W.data' * y.grad
  return nothing
end

# Dot

function primal!(z::GraphNode{:dot, 2})
  x, y = z.args
  z.data = dot(x.data, y.data)
  return nothing
end

function adjoint!(z::GraphNode{:dot, 2})
  x, y = z.args
  x.grad += y.data .* z.grad
  y.grad += x.data .* z.grad
  return nothing
end

# Dense

struct Dense <: Operator # Dense jest jednym z Operatorów
  insize  :: Int64
  outsize :: Int64
end

dense(pair :: Pair{Int64, Int64}) =
  Dense(first(pair), last(pair)) # umożliwia tworzenie warstwy Dense za pomocą pary: np. dense(2 => 16) wykonuje Dense(2, 16)
dense(pair :: Pair{Int64, Int64}, activation) =
  tuple(dense(pair), activation()) # umożliwia tworzenie warstwy wraz z funkcją aktywacji, np. dense(2 => 16, relu)

function (y::Dense)(x)
  n   = y.insize
  m   = y.outsize
  W   = GraphNode(randn(m, n) .* 0.01, true) 
  b   = GraphNode(randn(m) .* 0.01, true) 
  mul = GraphNode(:mul, (W, x), zeros(m))
  add = GraphNode(:add, (mul, b), zeros(m))
  return add
end

# Sigmoid

struct Sigmoid <: Operator end # Sigmoid jest jednym z operatorów
sigmoid() = Sigmoid() # uproszczenie zapisu konstruktora (z małej litery)

function (y::Sigmoid)(x)
  sz = length(x.data)
  return GraphNode(:sigmoid, (x,), zeros(sz))
end

function primal!(y::GraphNode{:sigmoid, 1})
  x, = y.args
  y.data = 1 ./ (1 .+ exp.(-x.data))
  return nothing
end

function adjoint!(y::GraphNode{:sigmoid, 1})
  x, = y.args
  x.grad += exp.(-x.data) ./ (1 .+ exp.(-x.data)) .^ 2 .* y.grad
  return nothing
end

# ReLU

struct ReLU <: Operator end # ReLU jest jednym z Operatorów
relu() = ReLU() # uproszczenie zapisu konstruktora (z małej litery)

function (y::ReLU)(x)
  sz  = length(x.data)
  return GraphNode(:relu, (x,), zeros(sz))
end

function primal!(y::GraphNode{:relu, 1})
  x, = y.args
  y.data .= max.(0, x.data)
  return nothing
end

function adjoint!(y::GraphNode{:relu, 1})
  x, = y.args
  for i in 1:length(x.data)
    if x.data[i] == y.data[i]
      x.grad[i] += y.grad[i]
	end
  end
  return nothing
end

# BinaryCrossEntropy (BCE)

struct BinaryCrossEntropy <: Loss end # BinaryCrossEntropy należy do Loss
bce(output, target) = BinaryCrossEntropy()(output, target) # skrócenie zapisu

function (E::BinaryCrossEntropy)(x, y)
  return GraphNode(:bce, (x, y), zeros(1))
end

function primal!(z::GraphNode{:bce, 2})
  x, y = z.args
  z.data = -(y.data .* log.(x.data) + (1 .- y.data) .* log.(1 .- x.data))
  return nothing
end

function adjoint!(z::GraphNode{:bce, 2})
  x, y = z.args
  x.grad -= y.data ./ x.data .* z.grad
  x.grad += (1 .- y.data) ./ (1 .- x.data) .* z.grad
  return nothing
end

# --------------------------Nowe--------------------------
# Flatten

struct Flatten <: Operator end
flatten() = Flatten()

function (y::Flatten)(x)
  # Calculate total features by multiplying W, H, C (everything except the last dimension B)
  features = prod(size(x.data)[1:end-1])
  B = size(x.data)[end]
  return GraphNode(:flatten, (x,), zeros(features, B))
end

function primal!(y::GraphNode{:flatten, 1})
  x, = y.args
  B = size(x.data)[end]
  # Reshape into a matrix where each column is one flattened image
  y.data .= reshape(x.data, :, B) 
  return nothing
end

function adjoint!(y::GraphNode{:flatten, 1})
  x, = y.args
  x.grad .+= reshape(y.grad, size(x.grad))
  return nothing
end

# Conv

struct Conv <: Operator
  filter_size :: NTuple{2, Int64}
  in_channels :: Int64
  out_channels :: Int64
  pad :: Int64
end

# Upraszczamy konstruktor, zakładamy bias=false zgodnie z Twoim zadaniem
function conv(filter_size::NTuple{2, Int64}, channels::Pair{Int64, Int64}; pad=0, bias=false)
  return Conv(filter_size, first(channels), last(channels), pad)
end

function (y::Conv)(x)
  W_in, H_in, _ = size(x.data) # Pobieramy szerokość i wysokość wejścia
  
  # Wzór na rozmiar wyjścia po konwolucji: W_out = W_in + 2*pad - filter_size + 1
  W_out = W_in + 2*y.pad - y.filter_size[1] + 1
  H_out = H_in + 2*y.pad - y.filter_size[2] + 1
  
  # Wagi filtru mają kształt: (Szerokość_filtru, Wysokość_filtru, Kanały_In, Kanały_Out)
  W_shape = (y.filter_size[1], y.filter_size[2], y.in_channels, y.out_channels)
  # Zgodnie z konwencją, inicjalizujemy małe losowe wagi
  W = GraphNode(randn(W_shape) .* 0.1, true) 
  
  return GraphNode(:conv, (W, x), zeros(W_out, H_out, y.out_channels))
end

function primal!(y::GraphNode{:conv, 2})
  W, x = y.args
  F_w, F_h, C_in, C_out = size(W.data)
  W_in, H_in, _ = size(x.data)
  W_out, H_out, _ = size(y.data)

  y.data .= 0 # Czyścimy wynik przed dodawaniem

  # Sprytny trick: wyliczamy, jaki pad był użyty, na podstawie wymiarów pudełek!
  pad_w = (W_out - W_in + F_w - 1) ÷ 2
  pad_h = (H_out - H_in + F_h - 1) ÷ 2

  # Tworzymy tymczasowy, powiększony o ramkę zer (padding) obrazek wejściowy
  x_padded = zeros(W_in + 2*pad_w, H_in + 2*pad_h, C_in)
  x_padded[pad_w+1:pad_w+W_in, pad_h+1:pad_h+H_in, :] .= x.data

  # Klasyczna konwolucja (okienko ślizga się po obrazku)
  for c_out in 1:C_out
    for c_in in 1:C_in
      for i in 1:W_out
        for j in 1:H_out
           # Wycinamy kawałek powiększonego obrazka pod okienkiem filtru
           patch = x_padded[i : i+F_w-1, j : j+F_h-1, c_in]
           filter = W.data[:, :, c_in, c_out]
           # Mnożymy liczby z okienka z liczbami z filtru i dodajemy wynik
           y.data[i, j, c_out] += sum(patch .* filter)
        end
      end
    end
  end
  return nothing
end

function adjoint!(y::GraphNode{:conv, 2})
  W, x = y.args
  F_w, F_h, C_in, C_out = size(W.data)
  W_in, H_in, _ = size(x.data)
  W_out, H_out, _ = size(y.data)

  pad_w = (W_out - W_in + F_w - 1) ÷ 2
  pad_h = (H_out - H_in + F_h - 1) ÷ 2

  x_padded = zeros(W_in + 2*pad_w, H_in + 2*pad_h, C_in)
  x_padded[pad_w+1:pad_w+W_in, pad_h+1:pad_h+H_in, :] .= x.data
  
  # Tymczasowa pusta tablica na błędy dla powiększonego obrazka
  dx_padded = zeros(size(x_padded))

  for c_out in 1:C_out
    for c_in in 1:C_in
      for i in 1:W_out
        for j in 1:H_out
           patch = x_padded[i : i+F_w-1, j : j+F_h-1, c_in]
           dy = y.grad[i, j, c_out] # Pojedynczy błąd, który przyszedł z góry

           # 1. Gradient po wagach (Błąd spada na okienko obrazka)
           W.grad[:, :, c_in, c_out] .+= patch .* dy

           # 2. Gradient po wejściu (Błąd spada na wagi filtru)
           filter = W.data[:, :, c_in, c_out]
           dx_padded[i : i+F_w-1, j : j+F_h-1, c_in] .+= filter .* dy
        end
      end
    end
  end

  # Na koniec odcinamy ramkę paddingu z błędów wejściowych (bo oryginalny x jej nie miał)
  x.grad .+= dx_padded[pad_w+1:pad_w+W_in, pad_h+1:pad_h+H_in, :]
  return nothing
end

# MaxPool

struct MaxPool <: Operator
  window :: NTuple{2, Int64}
end

maxpool(window::NTuple{2, Int64}) = MaxPool(window)

function (y::MaxPool)(x)
  W_in, H_in, C = size(x.data)
  # Obliczamy nowy rozmiar: np. 28 / 2 = 14
  W_out = W_in ÷ y.window[1]
  H_out = H_in ÷ y.window[2]
  return GraphNode(:maxpool, (x,), zeros(W_out, H_out, C))
end

function primal!(y::GraphNode{:maxpool, 1})
  x, = y.args
  W_out, H_out, C = size(y.data)
  
  # Sprytny trik: odtwarzamy rozmiar okienka z proporcji pudełek
  w_w = size(x.data, 1) ÷ W_out
  w_h = size(x.data, 2) ÷ H_out

  for c in 1:C
    for i in 1:W_out
      for j in 1:H_out
        # Wycinamy okienko 2x2 z obrazka i wybieramy największą wartość
        patch = x.data[(i-1)*w_w+1 : i*w_w, (j-1)*w_h+1 : j*w_h, c]
        y.data[i, j, c] = maximum(patch)
      end
    end
  end
  return nothing
end

function adjoint!(y::GraphNode{:maxpool, 1})
  x, = y.args
  W_out, H_out, C = size(y.data)
  
  w_w = size(x.data, 1) ÷ W_out
  w_h = size(x.data, 2) ÷ H_out

  for c in 1:C
    for i in 1:W_out
      for j in 1:H_out
        max_val = y.data[i, j, c]
        dy = y.grad[i, j, c]
        
        # Przekazujemy błąd (gradient) TYLKO do tego piksela, który był "zwycięzcą" (miał wartość max)
        for di in 1:w_w
          for dj in 1:w_h
            xi = (i-1)*w_w + di
            xj = (j-1)*w_h + dj
            if x.data[xi, xj, c] == max_val
              x.grad[xi, xj, c] += dy
            end
          end
        end
      end
    end
  end
  return nothing
end

# Dropout

struct Dropout <: Operator
  p :: Float64
end

dropout(p::Float64) = Dropout(p)

function (y::Dropout)(x)
  return GraphNode(:dropout, (x,), zeros(size(x.data)))
end

function primal!(y::GraphNode{:dropout, 1})
  x, = y.args
  y.data .= x.data # Przepuszczamy dane bez zmian
  return nothing
end

function adjoint!(y::GraphNode{:dropout, 1})
  x, = y.args
  x.grad .+= y.grad # Przepuszczamy błędy bez zmian
  return nothing
end

# LogitCrossEntropy

struct LogitCrossEntropy <: Loss end
logitcrossentropy(output, target) = LogitCrossEntropy()(output, target)

function (E::LogitCrossEntropy)(x, y)
  return GraphNode(:crossentropy, (x, y), zeros(1))
end

function primal!(z::GraphNode{:crossentropy, 2})
  x, y = z.args
  xmax = maximum(x.data)
  lse = xmax + log(sum(exp.(x.data .- xmax)))
  log_probs = x.data .- lse
  z.data[1] = -sum(y.data .* log_probs)
  return nothing
end

function adjoint!(z::GraphNode{:crossentropy, 2})
  x, y = z.args
  xmax = maximum(x.data)
  probs = exp.(x.data .- xmax) ./ sum(exp.(x.data .- xmax))
  x.grad .+= (probs .- y.data) .* z.grad[1]
  return nothing
end

## Uruchomienie biblioteki

In [ ]:
using MLDatasets, Flux
using Statistics: mean

println("\n[x] Ładowanie danych FashionMNIST...")
train_data = MLDatasets.FashionMNIST(split=:train) # 60 000
test_data  = MLDatasets.FashionMNIST(split=:test) # 10 000

function loader(data; batchsize::Int=1)
    x4dim = reshape(data.features, 28, 28, 1, :)
    yhot  = Flux.onehotbatch(data.targets, 0:9) # 10×60000 OneHotMatrix
    Flux.DataLoader((x4dim, yhot); batchsize, shuffle=true)
end

println("[x] Budowanie architektury modelu CNN...")
net = chain((
  conv((3, 3), 1 => 6, pad=1, bias=false),
  maxpool((2, 2)),
  conv((3, 3), 6 => 16, pad=1, bias=false),
  maxpool((2, 2)),
  flatten(),
  dense(784 => 84, relu),
  dropout(0.4),
  dense(84 => 10)
))


[x] Ładowanie danych FashionMNIST...
[x] Budowanie architektury modelu CNN...


20-element Vector{GraphNode}:
 weight
 weight
 weight
 weight
 tensor
 layer conv with 2 arg(s)
 layer maxpool with 1 arg(s)
 layer conv with 2 arg(s)
 layer maxpool with 1 arg(s)
 layer flatten with 1 arg(s)
 layer mul with 2 arg(s)
 weight
 layer add with 2 arg(s)
 layer relu with 1 arg(s)
 layer dropout with 1 arg(s)
 layer mul with 2 arg(s)
 weight
 layer add with 2 arg(s)
 tensor
 layer crossentropy with 2 arg(s)